In [ ]:
import matplotlib.pyplot as plt

class PCB:
    def __init__(self, pid, qos_name, qos_level, arrival_time, burst_time):
        self.pid = pid
        self.qos_name = qos_name
        self.priority = qos_level  # Higher level = Higher priority in iOS
        self.arrival_time = arrival_time
        self.burst_time = burst_time
        self.remaining_time = burst_time
        self.state = "New"
        self.waiting_time = 0
        self.turnaround_time = 0
        self.completion_time = 0
        self.io_wait_remaining = 0

def simulate_ios_scheduler():
    # Define tasks with iOS Quality of Service bands
    processes = [
        PCB("P1", "Utility", 2, 0, 4),
        PCB("P2", "UserInteractive", 4, 1, 3),
        PCB("P3", "UserInitiated", 3, 2, 2),
        PCB("P4", "Background", 1, 3, 3)
    ]
    
    time = 0
    running_proc = None
    gantt_log = []  # To store tuples of (pid, start_time, end_time)
    pipeline_history = []
    
    print("--- iOS Process State Transitions Step-by-Step ---")
    
    while True:
        # Check if all processes are completed
        if all(p.state == "Terminated" for p in processes):
            break
            
        # 1. Handle New Arrivals -> Move to Ready State
        for p in processes:
            if p.arrival_time == time and p.state == "New":
                p.state = "Ready"
                print(f"[Time {time}s] {p.pid} ({p.qos_name}) Created: New -> Ready")

        # 2. Handle Waiting State (Simulate an iOS I/O or background system call event)
        # Force P3 to yield for I/O when it encounters its first execution step at time 5
        if time == 5 and running_proc and running_proc.pid == "P3":
            running_proc.state = "Waiting"
            running_proc.io_wait_remaining = 2
            print(f"[Time {time}s] {running_proc.pid} requested resource: Running -> Waiting")
            running_proc = None

        # Decrement waiting counters for blocked processes
        for p in processes:
            if p.state == "Waiting":
                p.io_wait_remaining -= 1
                if p.io_wait_remaining == 0:
                    p.state = "Ready"
                    print(f"[Time {time+1}s] {p.pid} resource acquired: Waiting -> Ready")

        # 3. iOS Preemptive Scheduling Decision based on QoS priority level
        ready_pool = [p for p in processes if p.state == "Ready"]
        
        if ready_pool:
            highest_qos_proc = max(ready_pool, key=lambda x: x.priority)
            
            if running_proc:
                # Preemption check: iOS preempts if a higher QoS thread becomes ready
                if highest_qos_proc.priority > running_proc.priority:
                    print(f"[Time {time}s] Preemption! {highest_qos_proc.pid} preempts {running_proc.pid}")
                    running_proc.state = "Ready"
                    running_proc = highest_qos_proc
                    running_proc.state = "Running"
            else:
                running_proc = highest_qos_proc
                running_proc.state = "Running"
                print(f"[Time {time}s] Context Switch: {running_proc.pid} selected -> Running")

        # 4. Process execution tick
        if running_proc:
            gantt_log.append((running_proc.pid, time, time + 1))
            running_proc.remaining_time -= 1
            
            # Print state pipeline representation
            pipeline_str = f"Time {time:02d}s | " + " | ".join([f"{p.pid}: {p.state}" for p in processes])
            pipeline_history.append(pipeline_str)
            print(pipeline_str)
            
            # Check for completion
            if running_proc.remaining_time == 0:
                running_proc.state = "Terminated"
                running_proc.completion_time = time + 1
                running_proc.turnaround_time = running_proc.completion_time - running_proc.arrival_time
                running_proc.waiting_time = running_proc.turnaround_time - running_proc.burst_time
                print(f"[Time {time+1}s] {running_proc.pid} finished: Running -> Terminated")
                running_proc = None
        else:
            # Idle CPU step if everything is waiting
            pipeline_str = f"Time {time:02d}s | " + " | ".join([f"{p.pid}: {p.state}" for p in processes])
            pipeline_history.append(pipeline_str)
            print(pipeline_str)
            
        # Accumulate wait times for Ready processes
        for p in processes:
            if p.state == "Ready" and p != running_proc:
                p.waiting_time += 1
                
        time += 1

    # --- Calculations & Metrics Output ---
    print("\n" + "="*50)
    print("iOS CALCULATION METRICS")
    print("="*50)
    print(f"{'PID':<6}{'QoS Class':<18}{'Arrival':<10}{'Burst':<8}{'Turnaround':<12}{'Waiting':<8}")
    
    total_tat, total_wt = 0, 0
    for p in processes:
        total_tat += p.turnaround_time
        total_wt += p.waiting_time
        print(f"{p.pid:<6}{p.qos_name:<18}{p.arrival_time:<10}{p.burst_time:<8}{p.turnaround_time:<12}{p.waiting_time:<8}")
        
    print("-" * 50)
    print(f"Average Turnaround Time: {total_tat / len(processes):.2f}s")
    print(f"Average Waiting Time:    {total_wt / len(processes):.2f}s")
    
    # Generate Gantt Chart Visualisation
    generate_gantt_chart(gantt_log)

def generate_gantt_chart(gantt_log):
    fig, ax = plt.subplots(figsize=(10, 3))
    for entry in gantt_log:
        pid, start, end = entry
        color = 'tomato' if pid == 'P2' else 'skyblue' if pid == 'P3' else 'lightgreen' if pid == 'P1' else 'gold'
        ax.barh(y=pid, width=end-start, left=start, edgecolor='black', color=color)
        ax.text(start + (end-start)/2, pid, pid, ha='center', va='center', color='black', weight='bold')
        
    ax.set_xlabel('Timeline (Seconds)')
    ax.set_ylabel('Processes (PCB)')
    ax.set_title('iOS Preemptive QoS Scheduling Gantt Chart')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    simulate_ios_scheduler()